In [0]:
from pyspark.sql.functions import first, rand, count
dates = spark.sql("SELECT explode(sequence(DATE'2024-01-01', DATE'2024-03-24', INTERVAL 1 DAY)) as  calendar_date")
c_id = spark.sql("SELECT explode(sequence(1,200, 1)) as  client_id")
types = spark.sql("""SELECT concat("col_", colName) as col_name from (SELECT explode(sequence(1,20, 1)) as  colName)""")
 
dates = dates.repartition(99)
c_id = c_id.repartition(11)
types = types.repartition(1)
 
df_cartesian = c_id.crossJoin(dates.select("calendar_date")).crossJoin(types.select("col_name")).select("client_id","calendar_date","col_name")
df_cartesian2 = df_cartesian.groupBy("calendar_date").agg(count("client_id"))
 
# display(df_cartesian2.limit(1000))
 
df_cartesian = df_cartesian.withColumn("val", (rand()*10).cast("int"))

df_grp = df_cartesian.groupBy("client_id","calendar_date").pivot("col_name").agg((first("val").alias("val")))

display(df_grp.head(20))


client_id,calendar_date,col_1,col_10,col_11,col_12,col_13,col_14,col_15,col_16,col_17,col_18,col_19,col_2,col_20,col_3,col_4,col_5,col_6,col_7,col_8,col_9
170,2024-03-19,9,4,8,2,3,8,8,6,3,8,2,1,9,4,4,8,4,2,8,3
170,2024-03-07,3,8,1,9,3,8,7,1,7,4,2,8,5,8,1,2,9,7,8,1
170,2024-01-17,0,6,7,4,6,3,9,7,8,8,0,8,8,2,1,0,1,8,2,8
170,2024-01-31,5,5,9,5,0,7,4,4,2,3,1,6,5,4,5,2,4,1,7,1
170,2024-02-26,9,3,4,2,7,8,4,1,7,1,2,4,6,5,3,9,8,3,7,3
170,2024-03-21,4,7,3,3,1,4,2,8,5,0,6,3,7,3,1,6,6,4,4,3
170,2024-01-29,5,6,8,2,9,0,2,4,4,6,1,4,8,6,2,6,9,0,5,2
170,2024-03-24,0,2,5,8,7,5,0,6,1,2,6,6,4,9,0,8,4,2,8,7
170,2024-02-24,1,1,7,0,6,7,3,7,1,6,9,0,3,0,5,9,7,9,0,5
170,2024-02-29,5,2,5,5,6,4,4,6,3,7,2,3,4,7,0,3,8,6,5,2


## INNER JOIN

In [0]:

inner_join=df_cartesian.join(df_cartesian2, df_cartesian.calendar_date==df_cartesian2.calendar_date, 'inner')
display(inner_join.head(10))
     

client_id,calendar_date,col_name,val,calendar_date,count(client_id)
170,2024-03-19,col_1,9,2024-03-19,4000
170,2024-03-19,col_2,3,2024-03-19,4000
170,2024-03-19,col_3,0,2024-03-19,4000
170,2024-03-19,col_4,5,2024-03-19,4000
170,2024-03-19,col_5,9,2024-03-19,4000
170,2024-03-19,col_6,4,2024-03-19,4000
170,2024-03-19,col_7,5,2024-03-19,4000
170,2024-03-19,col_8,0,2024-03-19,4000
170,2024-03-19,col_9,1,2024-03-19,4000
170,2024-03-19,col_10,5,2024-03-19,4000


## LEFT JOIN 

In [0]:

left_join=df_cartesian.join(df_cartesian2, df_cartesian.calendar_date==df_cartesian2.calendar_date, 'left')
display(left_join.head(10))

client_id,calendar_date,col_name,val,calendar_date,count(client_id)
170,2024-03-19,col_1,9,2024-03-19,4000
170,2024-03-19,col_2,3,2024-03-19,4000
170,2024-03-19,col_3,0,2024-03-19,4000
170,2024-03-19,col_4,5,2024-03-19,4000
170,2024-03-19,col_5,9,2024-03-19,4000
170,2024-03-19,col_6,4,2024-03-19,4000
170,2024-03-19,col_7,5,2024-03-19,4000
170,2024-03-19,col_8,0,2024-03-19,4000
170,2024-03-19,col_9,1,2024-03-19,4000
170,2024-03-19,col_10,5,2024-03-19,4000


## Duplikaty 1 rozwiązanie

Usunięcie podwojonej kolumny po połączeniu(after the join). Podczas
łączenia użyj oryginalnych Dataframes

In [0]:
inner_join = df_cartesian.join(df_cartesian2, df_cartesian.calendar_date == df_cartesian2.calendar_date, 'inner')

# Usuwamy zdublowaną kolumnę (np. calendar_date z drugiego DataFrame’u)
inner_clean = inner_join.drop(df_cartesian2.calendar_date)
inner_clean = inner_clean.dropDuplicates()

display(inner_clean.limit(10))

client_id,col_name,val,calendar_date,count(client_id)
192,col_6,8,2024-01-09,4000
192,col_6,6,2024-02-28,4000
192,col_6,7,2024-01-12,4000
192,col_6,1,2024-03-04,4000
192,col_6,6,2024-02-11,4000
192,col_6,5,2024-03-03,4000
192,col_6,8,2024-02-09,4000
192,col_6,4,2024-03-20,4000
192,col_6,8,2024-01-08,4000
192,col_6,2,2024-01-11,4000


In [0]:
left_join = df_cartesian.join(df_cartesian2, df_cartesian.calendar_date == df_cartesian2.calendar_date, 'left')
left_clean = left_join.drop(df_cartesian2.calendar_date).dropDuplicates()
display(left_clean.limit(10))

client_id,col_name,val,calendar_date,count(client_id)
170,col_1,9,2024-03-19,4000
170,col_1,3,2024-03-07,4000
170,col_1,0,2024-01-17,4000
170,col_1,5,2024-01-31,4000
170,col_1,9,2024-02-26,4000
170,col_1,4,2024-03-21,4000
170,col_1,5,2024-01-29,4000
170,col_1,0,2024-03-24,4000
170,col_1,1,2024-02-24,4000
170,col_1,5,2024-02-29,4000


## Duplikaty - 2 rozwiązanie 

Zmiana nazw kolumn przed połączeniem

In [0]:
# Zmieniamy nazwę kolumny w jednym z DataFrame'ów
df_cartesian2_renamed = df_cartesian2.withColumnRenamed("calendar_date", "calendar_date_2")

# Łączenie przez warunek logiczny, ale bez kolizji nazw
inner_join = df_cartesian.join(df_cartesian2_renamed, df_cartesian["calendar_date"] == df_cartesian2_renamed["calendar_date_2"], "inner")

inner_clean = inner_join.dropDuplicates()
display(inner_clean.limit(10))

client_id,calendar_date,col_name,val,calendar_date_2,count(client_id)
192,2024-01-09,col_6,8,2024-01-09,4000
192,2024-02-28,col_6,6,2024-02-28,4000
192,2024-01-12,col_6,7,2024-01-12,4000
192,2024-03-04,col_6,1,2024-03-04,4000
192,2024-02-11,col_6,6,2024-02-11,4000
192,2024-03-03,col_6,5,2024-03-03,4000
192,2024-02-09,col_6,8,2024-02-09,4000
192,2024-03-20,col_6,4,2024-03-20,4000
192,2024-01-08,col_6,8,2024-01-08,4000
192,2024-01-11,col_6,2,2024-01-11,4000


In [0]:
left_join = df_cartesian.join(df_cartesian2_renamed, df_cartesian["calendar_date"] == df_cartesian2_renamed["calendar_date_2"], "left")
left_clean = left_join.dropDuplicates()
display(left_clean.limit(10))

client_id,calendar_date,col_name,val,calendar_date_2,count(client_id)
192,2024-01-09,col_6,8,2024-01-09,4000
192,2024-02-28,col_6,6,2024-02-28,4000
192,2024-01-12,col_6,7,2024-01-12,4000
192,2024-03-04,col_6,1,2024-03-04,4000
192,2024-02-11,col_6,6,2024-02-11,4000
192,2024-03-03,col_6,5,2024-03-03,4000
192,2024-02-09,col_6,8,2024-02-09,4000
192,2024-03-20,col_6,4,2024-03-20,4000
192,2024-01-08,col_6,8,2024-01-08,4000
192,2024-01-11,col_6,2,2024-01-11,4000
